## BNN MNIST: Hardware vs Fast Software Model

This notebook imports the `BNN_MNIST` class from `bnn_mnist.py` and uses the fast batch inference function to verify the FPGA output.

In [ ]:
from pynq import Overlay, allocate
import numpy as np
import time

from bnn_mnist import BNN_MNIST

In [ ]:
bnn_sw = BNN_MNIST()
print("Software model loaded successfully.")

In [ ]:
mnist = np.load("dataset/mnist_test_data_original.npy", allow_pickle=True)
X_test = mnist.item().get("data")
y_test = mnist.item().get("label")
X_test = np.reshape(X_test, (10000, 784))

NUM_SAMPLES = 1000
X_batch = X_test[:NUM_SAMPLES]
y_batch = y_test[:NUM_SAMPLES]

In [ ]:
# --- Run Fast Software Inference ---
start_time = time.time()

# Call the new function you added
sw_predictions = bnn_sw.inference_batch(X_batch)

end_time = time.time()
print(f"Software inference for {NUM_SAMPLES} images took {end_time - start_time:.4f}s")

In [ ]:
# --- Run Hardware Inference ---
ol = Overlay('design_1.bit')
dma = ol.axi_dma_0

# Packing Logic (Required for Hardware)
def pack_batch(images):
    # Applies binarization and packing to 25x uint32 format
    num = len(images)
    packed_buffer = allocate(shape=(num, 25), dtype=np.uint32)
    
    # Reuse bnn_sw helpers for consistency
    for i in range(num):
        # Get binarized bipolar input
        img_bi = bnn_sw.sign(bnn_sw.adj(images[i]))
        # Pad to 800 bits (add 16 ones)
        img_padded = np.append(img_bi, [1] * 16)
        # Pack
        packed_buffer[i] = bnn_sw.pack(img_padded, 800)
    return packed_buffer

in_buffer = pack_batch(X_batch)
out_buffer = allocate(shape=(NUM_SAMPLES,), dtype=np.int32)

dma.sendchannel.transfer(in_buffer)
dma.recvchannel.transfer(out_buffer)
dma.sendchannel.wait()
dma.recvchannel.wait()

In [ ]:
# --- Compare Results ---
matches = 0
print(f"{'Index':<8} {'HW Pred':<10} {'SW Pred':<10} {'Label':<8} {'Status'}")
print("-"*50)

for i in range(NUM_SAMPLES):
    hw = out_buffer[i]
    sw = sw_predictions[i]
    lbl = y_batch[i]
    
    if hw == sw:
        matches += 1
        
    if i < 10: # Print first 10
        status = "OK" if hw == sw else "MISMATCH"
        print(f"{i:<8} {hw:<10} {sw:<10} {lbl:<8} {status}")

print("-"*50)
print(f"Total Matches (HW vs SW): {matches}/{NUM_SAMPLES}")
if matches == NUM_SAMPLES:
    print("SUCCESS: Hardware matches Software Golden Model perfectly.")
else:
    print("FAILURE: Hardware does not match Software.")